In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing_extensions import TypedDict

load_dotenv()

True

In [13]:
parent_llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2)
sub_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [14]:
class parent_state(TypedDict):
    question: str
    answer_eng: str
    answer_hin: str

In [15]:
def translate_text(state: parent_state):
    prompt = f"""
    
    you are a hindi translate assistnat and you need to just transtat the text coming from answer_eng into hindi
    
    text: {state['answer_eng']}
    
    """
    answer = sub_llm.invoke(prompt).content
    return {'answer_hin': answer}

In [16]:
# graph of sub 
sub = StateGraph(parent_state)

sub.add_node("translate_text", translate_text)

sub.add_edge(START, "translate_text")
sub.add_edge("translate_text", END)

subgraph = sub.compile()

In [17]:
def generate_text(state: parent_state):
    answer = parent_llm.invoke(f"you are a helpful assistant, answer this question {state['question']}").content
    return {'answer_eng': answer}

In [18]:
# parent graph
parent = StateGraph(parent_state)

parent.add_node("generate_text", generate_text)
parent.add_node("translate_text", subgraph)

parent.add_edge(START, "generate_text")
parent.add_edge("generate_text", "translate_text")
parent.add_edge("translate_text", END)

graph = parent.compile()

In [19]:
graph.invoke({"question": "write a 10 line para for assassins creed shadows game"})

{'question': 'write a 10 line para for assassins creed shadows game',
 'answer_eng': "Assassin's Creed: Shadows is a mobile strategy game that brings the iconic Brotherhood into a new, accessible format.  \nSet in the 15th‑century Italian Renaissance, players assume the role of a young assassin tasked with protecting the city from the oppressive Templar forces.  \nThe game blends turn‑based tactical combat with real‑time strategy, allowing players to command a squad of assassins across a sprawling, interactive map.  \nEach character possesses unique abilities and skill trees, encouraging thoughtful team composition and strategic upgrades.  \nPlayers can recruit legendary figures such as Ezio Auditore and Altaïr Ibn-LaʼAhad, adding depth and nostalgia to the roster.  \nResource management is key: gathering gold, recruiting allies, and upgrading weapons all play a part in sustaining your Brotherhood’s influence.  \nThe narrative unfolds through cinematic cutscenes and dialogue, weaving a